
# Cento Clausula Frequency Analysis

Compare the frequency of 3- and 4-token clausulae (metrical line endings) in
Ausonius's *Cento Nuptialis* against the combined Aeneid and Georgics of Vergil.

Each hexameter line is represented as a sequence of prime-encoded metrical
numbers ending with `<EOL>`.  A **3-token clausula** is the last 2 numeric
tokens + `<EOL>` (e.g. `18 4 <EOL>`), and a **4-token clausula** is the last
3 numeric tokens + `<EOL>` (e.g. `2 18 4 <EOL>`).

## Method

For each clausula type, a one-sided **Fisher's exact test** is used to test
whether the clausula is over-represented in the cento relative to Vergil.
Results are sorted by p-value. The odds ratio gives a measure of effect size.

For the top over-represented clausulae (count > 1 in the cento), we also
check whether the same clausulae are common in the full Ausonius corpus
(excluding the cento) vs Vergil, to distinguish Ausonius's general style
from cento-specific patterns.


In [1]:

from __future__ import annotations

from collections import Counter
from pathlib import Path

import pandas as pd
from scipy.stats import fisher_exact


In [2]:

def extract_clausulae_from_file(path: Path, n: int) -> list[tuple[int, ...]]:
    """Extract N-token clausulae from a mapped_lines file.

    Each line is expected to end with numeric tokens followed by <EOL>.
    The clausula is the last N numeric tokens before <EOL>.

    Returns a list of tuples, one per line.
    """
    clausulae: list[tuple[int, ...]] = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split()
            # Strip trailing <EOL>
            if parts and parts[-1] == "<EOL>":
                parts = parts[:-1]
            # Convert to int
            try:
                nums = [int(p) for p in parts]
            except ValueError:
                continue
            if len(nums) >= n:
                clausulae.append(tuple(nums[-n:]))
    return clausulae


def load_corpus_clausulae(data_dir: Path, n: int) -> list[tuple[int, ...]]:
    """Load all mapped_lines files from a directory and extract N-token clausulae."""
    all_clausulae: list[tuple[int, ...]] = []
    for path in sorted(data_dir.glob("*.mapped_lines.txt")):
        all_clausulae.extend(extract_clausulae_from_file(path, n))
    return all_clausulae


def clausula_frequencies(
    clausulae: list[tuple[int, ...]],
) -> Counter[tuple[int, ...]]:
    """Count clausula frequencies."""
    return Counter(clausulae)


def build_comparison_table(
    cento_clausulae: list[tuple[int, ...]],
    vergil_clausulae: list[tuple[int, ...]],
    n: int,
) -> pd.DataFrame:
    """Build a comparison table using Fisher's exact test.

    For each clausula, runs a one-sided Fisher's exact test to determine
    whether it is over-represented in the cento relative to Vergil.

    Returns DataFrame sorted by p-value (most over-represented first).
    """
    cento_freq = clausula_frequencies(cento_clausulae)
    vergil_freq = clausula_frequencies(vergil_clausulae)

    cento_total = len(cento_clausulae)
    vergil_total = len(vergil_clausulae)

    # All clausulae that appear in either corpus
    all_clausulae = sorted(set(cento_freq.keys()) | set(vergil_freq.keys()))

    rows = []
    for cl in all_clausulae:
        c_count = cento_freq.get(cl, 0)
        v_count = vergil_freq.get(cl, 0)
        c_pct = (c_count / cento_total * 100) if cento_total else 0.0
        v_pct = (v_count / vergil_total * 100) if vergil_total else 0.0

        # 2x2 contingency table for Fisher's exact test
        a = c_count
        b = v_count
        c = cento_total - c_count
        d = vergil_total - v_count
        table = [[a, b], [c, d]]

        odds_ratio, p_greater = fisher_exact(table, alternative="greater")

        rows.append({
            "clausula": " ".join(str(x) for x in cl),
            "cento_count": c_count,
            "cento_pct": round(c_pct, 3),
            "vergil_count": v_count,
            "vergil_pct": round(v_pct, 3),
            "odds_ratio": round(odds_ratio, 3),
            "p_overrep": p_greater,
        })

    df = pd.DataFrame(rows)

    # Sort by p-value (most significantly over-represented first)
    df = df.sort_values("p_overrep", ascending=True).reset_index(drop=True)

    return df



## Load Data

Extract 3- and 4-token clausulae from the cento (question) and the combined
Aeneid + Georgics (Vergil).


In [3]:

# Clausula sizes: 3-token and 4-token INCLUDING the <EOL> marker
# So we extract 2 and 3 numeric tokens respectively (the <EOL> is implicit)
CLAUSULA_SIZES = [2, 3]

# Load cento clausulae
cento_2 = load_corpus_clausulae(Path("metrical_model_lines/question"), 2)
cento_3 = load_corpus_clausulae(Path("metrical_model_lines/question"), 3)

# Load Vergil (Georgics + Aeneid) clausulae
vergil_2 = load_corpus_clausulae(Path("metrical_model_lines/vergil"), 2)
vergil_2 += load_corpus_clausulae(Path("metrical_model_lines/aeneid"), 2)
vergil_3 = load_corpus_clausulae(Path("metrical_model_lines/vergil"), 3)
vergil_3 += load_corpus_clausulae(Path("metrical_model_lines/aeneid"), 3)

print(f"Cento: {len(cento_2)} lines (3-token clausula), {len(cento_3)} lines (4-token clausula)")
print(f"Vergil: {len(vergil_2)} lines (3-token clausula), {len(vergil_3)} lines (4-token clausula)")


Cento: 131 lines (3-token clausula), 131 lines (4-token clausula)
Vergil: 12079 lines (3-token clausula), 12060 lines (4-token clausula)



## Clausula Frequency Comparison

For each clausula size, run a one-sided **Fisher's exact test** to determine
whether the clausula is over-represented in the cento relative to Vergil.

Results are sorted by p-value (most over-represented first). The odds ratio
gives a measure of effect size.

3-token clausulae have 2 numeric tokens + `<EOL>`; 4-token clausulae have
3 numeric tokens + `<EOL>`.


In [4]:

def fisher_pvalue(c_count: int, c_total: int, v_count: int, v_total: int) -> tuple[float, float]:
    """One-sided Fisher's exact test for over-representation in cento/first corpus."""
    table = [[c_count, v_count], [c_total - c_count, v_total - v_count]]
    odds_ratio, p = fisher_exact(table, alternative="greater")
    return odds_ratio, p


# Build tables for 3- and 4-token clausulae (including <EOL>)
print("=" * 80)
print("3-TOKEN CLAUSULAE (2 numeric tokens + <EOL>)")
print("=" * 80)
df_3tok = build_comparison_table(cento_2, vergil_2, 2)
print(f"\nTotal 3-token clausulae: cento={len(cento_2)}, vergil={len(vergil_2)}")
print(f"Unique: cento={len(set(cento_2))}, vergil={len(set(vergil_2))}")

print("\n" + "=" * 80)
print("4-TOKEN CLAUSULAE (3 numeric tokens + <EOL>)")
print("=" * 80)
df_4tok = build_comparison_table(cento_3, vergil_3, 3)
print(f"\nTotal 4-token clausulae: cento={len(cento_3)}, vergil={len(vergil_3)}")
print(f"Unique: cento={len(set(cento_3))}, vergil={len(set(vergil_3))}")

# Load Ausonius (excluding cento) clausulae for comparison
ausonius_2 = load_corpus_clausulae(Path("metrical_model_lines/ausonius"), 2)
ausonius_3 = load_corpus_clausulae(Path("metrical_model_lines/ausonius"), 3)
print(f"\nAusonius (excl. cento): {len(ausonius_2)} lines (3-token), {len(ausonius_3)} lines (4-token)")


def show_top_clausulae_with_ausonius(
    df: pd.DataFrame,
    cento_clausulae: list[tuple[int, ...]],
    vergil_clausulae: list[tuple[int, ...]],
    ausonius_clausulae: list[tuple[int, ...]],
    label: str,
) -> None:
    """Show top 10 over-represented clausulae (count > 1) with Ausonius comparison."""
    cento_total = len(cento_clausulae)
    vergil_total = len(vergil_clausulae)
    ausonius_total = len(ausonius_clausulae)

    cento_freq = clausula_frequencies(cento_clausulae)
    vergil_freq = clausula_frequencies(vergil_clausulae)
    ausonius_freq = clausula_frequencies(ausonius_clausulae)

    # Top 10 by p-value, count > 1 in cento
    top = df[(df["cento_count"] > 1)].head(10)

    rows = []
    for _, row in top.iterrows():
        cl_parts = tuple(int(x) for x in row["clausula"].split())
        c_count = cento_freq.get(cl_parts, 0)
        v_count = vergil_freq.get(cl_parts, 0)
        a_count = ausonius_freq.get(cl_parts, 0)
        c_pct = (c_count / cento_total * 100) if cento_total else 0.0
        v_pct = (v_count / vergil_total * 100) if vergil_total else 0.0
        a_pct = (a_count / ausonius_total * 100) if ausonius_total else 0.0

        # Fisher: cento vs Vergil
        or_cento, p_cento = fisher_pvalue(c_count, cento_total, v_count, vergil_total)
        # Fisher: Ausonius vs Vergil
        or_aus, p_aus = fisher_pvalue(a_count, ausonius_total, v_count, vergil_total)

        rows.append({
            "clausula": row["clausula"],
            "cento_count": c_count,
            "cento_pct": round(c_pct, 3),
            "vergil_count": v_count,
            "vergil_pct": round(v_pct, 3),
            "ausonius_count": a_count,
            "ausonius_pct": round(a_pct, 3),
            "OR_cento": round(or_cento, 2),
            "p_cento": p_cento,
            "OR_ausonius": round(or_aus, 2),
            "p_ausonius": p_aus,
        })

    result = pd.DataFrame(rows)
    print(f"\n{label} — Top 10 over-represented (count > 1 in cento), with Ausonius comparison")
    print("=" * 140)
    display(
        result.style.format({
            "cento_pct": "{:.3f}%",
            "vergil_pct": "{:.3f}%",
            "ausonius_pct": "{:.3f}%",
            "p_cento": "{:.2e}",
            "p_ausonius": "{:.2e}",
        })
    )


show_top_clausulae_with_ausonius(df_3tok, cento_2, vergil_2, ausonius_2, "3-TOKEN CLAUSULAE")
show_top_clausulae_with_ausonius(df_4tok, cento_3, vergil_3, ausonius_3, "4-TOKEN CLAUSULAE")


3-TOKEN CLAUSULAE (2 numeric tokens + <EOL>)

Total 3-token clausulae: cento=131, vergil=12079
Unique: cento=17, vergil=118

4-TOKEN CLAUSULAE (3 numeric tokens + <EOL>)

Total 4-token clausulae: cento=131, vergil=12060
Unique: cento=51, vergil=393

Ausonius (excl. cento): 2247 lines (3-token), 2247 lines (4-token)

3-TOKEN CLAUSULAE — Top 10 over-represented (count > 1 in cento), with Ausonius comparison


,clausula,cento_count,cento_pct,vergil_count,vergil_pct,ausonius_count,ausonius_pct,OR_cento,p_cento,OR_ausonius,p_ausonius
0,132 12,5,3.817%,129,1.068%,8,0.356%,3.680000,1.46e-02,0.330000,1.00e+00
1,54 12,10,7.634%,598,4.951%,106,4.717%,1.590000,1.18e-01,0.950000,6.97e-01
2,18 12,2,1.527%,97,0.803%,33,1.469%,1.920000,2.87e-01,1.840000,2.79e-03
3,24 12,2,1.527%,108,0.894%,30,1.335%,1.720000,3.31e-01,1.500000,3.64e-02
4,18 4,42,32.061%,3727,30.855%,543,24.166%,1.060000,4.16e-01,0.710000,1.00e+00
5,12 12,23,17.557%,2008,16.624%,249,11.081%,1.070000,4.24e-01,0.630000,1.00e+00
6,6 12,19,14.504%,1775,14.695%,390,17.356%,0.980000,5.63e-01,1.220000,7.94e-04
7,36 4,13,9.924%,1356,11.226%,214,9.524%,0.870000,7.21e-01,0.830000,9.93e-01
8,54 4,2,1.527%,273,2.260%,81,3.605%,0.670000,7.98e-01,1.620000,2.10e-04
9,3 4,6,4.580%,795,6.582%,95,4.228%,0.680000,8.67e-01,0.630000,1.00e+00



4-TOKEN CLAUSULAE — Top 10 over-represented (count > 1 in cento), with Ausonius comparison


,clausula,cento_count,cento_pct,vergil_count,vergil_pct,ausonius_count,ausonius_pct,OR_cento,p_cento,OR_ausonius,p_ausonius
0,3 6 12,3,2.290%,37,0.307%,5,0.223%,7.620000,8.97e-03,0.720000,8.11e-01
1,45 24 12,2,1.527%,14,0.116%,3,0.134%,13.340000,1.25e-02,1.150000,5.13e-01
2,4 132 12,3,2.290%,82,0.680%,3,0.134%,3.420000,6.35e-02,0.200000,1.00e+00
3,9 18 4,9,6.870%,517,4.287%,111,4.940%,1.650000,1.13e-01,1.160000,9.29e-02
4,18 54 12,3,2.290%,132,1.095%,30,1.335%,2.120000,1.77e-01,1.220000,1.88e-01
5,18 12 12,8,6.107%,501,4.154%,62,2.759%,1.500000,1.82e-01,0.650000,1.00e+00
6,20 12 12,2,1.527%,74,0.614%,4,0.178%,2.510000,1.97e-01,0.290000,9.99e-01
7,10 18 4,2,1.527%,86,0.713%,8,0.356%,2.160000,2.44e-01,0.500000,9.86e-01
8,6 54 12,2,1.527%,89,0.738%,23,1.024%,2.090000,2.56e-01,1.390000,1.03e-01
9,6 36 4,3,2.290%,162,1.343%,42,1.869%,1.720000,2.61e-01,1.400000,3.67e-02



## Top Over-Represented Clausulae: Cento vs Vergil, and Ausonius vs Vergil

For the top 10 most over-represented clausulae (by Fisher's exact test) that
appear more than once in the cento, we also check whether the same clausulae
are more common in the full Ausonius corpus (excluding the cento) vs Vergil.
This helps distinguish patterns that are Ausonius's general style from those
specific to the cento.



## Full Frequency Tables

Complete sorted tables for manual inspection, ordered by Fisher's exact test
p-value (most over-represented in the cento first).


In [5]:

# Full tables: all clausulae in the cento, sorted by p-value
print("3-TOKEN CLAUSULAE — Full table (clausulae in cento, sorted by p-value)")
print("=" * 100)
display(
    df_3tok[df_3tok["cento_count"] > 0][
        ["clausula", "cento_count", "cento_pct", "vergil_count", "vergil_pct",
         "odds_ratio", "p_overrep"]
    ].style.format({
        "cento_pct": "{:.3f}%",
        "vergil_pct": "{:.3f}%",
        "p_overrep": "{:.2e}",
    })
)

print()
print("4-TOKEN CLAUSULAE — Full table (clausulae in cento, sorted by p-value)")
print("=" * 100)
display(
    df_4tok[df_4tok["cento_count"] > 0][
        ["clausula", "cento_count", "cento_pct", "vergil_count", "vergil_pct",
         "odds_ratio", "p_overrep"]
    ].style.format({
        "cento_pct": "{:.3f}%",
        "vergil_pct": "{:.3f}%",
        "p_overrep": "{:.2e}",
    })
)


3-TOKEN CLAUSULAE — Full table (clausulae in cento, sorted by p-value)


,clausula,cento_count,cento_pct,vergil_count,vergil_pct,odds_ratio,p_overrep
0,132 12,5,3.817%,129,1.068%,3.676000,1.46e-02
1,4 396,1,0.763%,8,0.066%,11.607000,9.25e-02
2,54 12,10,7.634%,598,4.951%,1.587000,1.18e-01
3,324 4,1,0.763%,14,0.116%,6.629000,1.49e-01
4,594 12,1,0.763%,20,0.166%,4.638000,2.03e-01
5,18 12,2,1.527%,97,0.803%,1.915000,2.87e-01
6,72 4,1,0.763%,35,0.290%,2.647000,3.22e-01
7,24 12,2,1.527%,108,0.894%,1.718000,3.31e-01
8,2 2,1,0.763%,43,0.356%,2.153000,3.78e-01
9,18 4,42,32.061%,3727,30.855%,1.058000,4.16e-01



4-TOKEN CLAUSULAE — Full table (clausulae in cento, sorted by p-value)


,clausula,cento_count,cento_pct,vergil_count,vergil_pct,odds_ratio,p_overrep
0,3 6 12,3,2.290%,37,0.307%,7.616000,8.97e-03
1,45 24 12,2,1.527%,14,0.116%,13.340000,1.25e-02
2,18 4 396,1,0.763%,1,0.008%,92.762000,2.14e-02
3,110 6 12,1,0.763%,3,0.025%,30.915000,4.23e-02
4,54 2 2,1,0.763%,3,0.025%,30.915000,4.23e-02
5,9 324 4,1,0.763%,3,0.025%,30.915000,4.23e-02
6,66 12 12,1,0.763%,3,0.025%,30.915000,4.23e-02
7,9 72 4,1,0.763%,5,0.041%,18.546000,6.28e-02
8,4 132 12,3,2.290%,82,0.680%,3.424000,6.35e-02
9,4 594 12,1,0.763%,8,0.066%,11.588000,9.27e-02
